In [2]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = True

In [3]:
@dataclass
class Message:
    content: str

In [4]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost


host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

In [5]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [6]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

In [8]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [9]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:
    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [10]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [11]:
display(Markdown(response.content))

## Pros of AutoGen:
Here are some compelling reasons to consider using AutoGen for your new AI Agent project:

1. **Multi-Agent Collaboration**: AutoGen allows multiple AI agents to work together, which can enhance performance on complex tasks compared to relying on a single large language model (LLM).

2. **Modularity**: The framework supports a modular approach to task management. You can break down complex tasks into manageable parts, potentially improving efficiency and scalability.

3. **Strong Backing**: Being developed within Microsoft Research, AutoGen benefits from substantial resources and support, ensuring robustness and reliability in various usage scenarios.

4. **Integration Capabilities**: AutoGen can seamlessly integrate with various LLMs, tools, and human inputs, making it versatile for different applications.

5. **Open Source**: The open-source nature of AutoGen enables customization and flexibility, allowing developers to tailor the framework to meet specific project needs.

6. **Performance and Automation**: By utilizing multiple agents and task specialization, AutoGen can drive real business value through effective automation and streamlined operations.

7. **Funding Stability**: With financial backing from Microsoft, AutoGen is less likely to face revenue pressures, indicating a focus on long-term development and stability.

These pros make AutoGen an attractive choice for developing AI agent applications that require adaptability, modularity, and collaborative capabilities.

TERMINATE

## Cons of AutoGen:
Here are some cons of using AutoGen in an AI Agent project:

1. **Steep Learning Curve**: AutoGen has a learning curve that can be challenging for new users, making it less accessible for teams without extensive technical expertise.

2. **Complex Documentation**: The documentation is often criticized for being hard to read and lacking sufficient examples, which can hinder effective implementation and troubleshooting.

3. **Inconsistent Outputs**: Users have reported experiencing inconsistent outputs from the system, leading to potential reliability issues in critical applications.

4. **Debugging Difficulties**: The debugging process with AutoGen can be more complicated compared to other frameworks, potentially increasing development time and costs.

5. **Fragile Behavior**: AutoGen can exhibit fragile behavior, meaning that it may not perform reliably under varying conditions or when applied to tasks it wasn't explicitly designed for.

6. **Higher Costs**: Operating costs may rise due to the potential need for more hands-on coding and troubleshooting compared to more user-friendly alternatives.

These factors should be considered when deciding whether to implement AutoGen in your AI project. 

TERMINATE



## Decision:

Based on the outlined pros and cons, I would recommend adopting AutoGen for the project. The compelling advantages, particularly its support for multi-agent collaboration, modularity, strong backing by Microsoft, and integration capabilities, outweigh the drawbacks. While the steep learning curve and documentation challenges may initial pose hurdles, the long-term benefits of improved efficiency, automation, and stability are significant. The ability to customize and adapt the framework to specific project needs further bolsters this decision.

TERMINATE

In [12]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [13]:
await host.stop()